# Orbit Wars ROI Reserve Agent v1

## Goal

Package and evaluate the first non-starter Orbit Wars agent. This candidate improves on the official nearest-target starter with four conservative guards:

- **Target ROI:** prefer production value adjusted for capture cost and travel time.
- **Source reserve:** keep ships on owned planets before launching.
- **Sun-path rejection:** skip direct launch paths that cross the central sun.
- **Action safety:** return plain `int` / `float` list values and fail closed with `[]`.

## Benchmark Contract

Run the same **30 fixed seeds** used by the EDA baseline against `random`. The first comparison target is the starter baseline: `23` wins, `7` losses, and `0` run errors.

## Artifacts Written

- `main.py`
- `submission.tar.gz`
- `agent_v1_environment_info.json`
- `agent_v1_seed_summary.csv`
- `agent_v1_run_errors.csv`
- `agent_v1_findings.md`

## 1. Runtime Setup

Locate the canonical `main.py`, copy it to the working directory, and define the benchmark configuration. Kaggle output should always contain a root-level `main.py` for submission.

In [ ]:
import importlib.util
import json
import platform
import shutil
import sys
import tarfile
import traceback
from pathlib import Path
from typing import Any, Optional, Sequence

import pandas as pd

EMBEDDED_AGENT_SOURCE = '"""ROI, reserve, and sun-safe Orbit Wars agent v1."""\n\nimport math\nfrom collections import namedtuple\nfrom typing import Any, Iterable, Optional\n\ntry:\n    from kaggle_environments.envs.orbit_wars.orbit_wars import Planet\nexcept Exception:\n    Planet = namedtuple("Planet", "id owner x y radius ships production")\n\n\nBOARD_CENTER = (50.0, 50.0)\nSUN_RADIUS = 10.0\nSUN_SAFETY_MARGIN = 0.25\nMAX_FLEET_SPEED = 6.0\n\n\ndef get_obs_value(obs: Any, name: str, default: Any) -> Any:\n    """Read an observation field from either dict or object observations.\n\n    Args:\n        obs: Kaggle observation as a dict-like or object-like value.\n        name: Field name to read.\n        default: Value returned when the field is absent.\n\n    Returns:\n        The observation field value, or `default` when unavailable.\n    """\n    if isinstance(obs, dict):\n        return obs.get(name, default)\n    return getattr(obs, name, default)\n\n\ndef parse_planets(rows: Iterable[Iterable[Any]]) -> list:\n    """Convert raw planet rows to Orbit Wars planet tuples.\n\n    Args:\n        rows: Iterable of `[id, owner, x, y, radius, ships, production]`.\n\n    Returns:\n        Parsed planet records with attribute access.\n    """\n    return [Planet(*row) for row in rows]\n\n\ndef distance(source: Any, target: Any) -> float:\n    """Calculate Euclidean distance between two planet-like objects."""\n    return math.hypot(float(target.x) - float(source.x), float(target.y) - float(source.y))\n\n\ndef launch_angle(source: Any, target: Any) -> float:\n    """Calculate launch angle from source to target in radians."""\n    return float(math.atan2(float(target.y) - float(source.y), float(target.x) - float(source.x)))\n\n\ndef fleet_speed(ships: int, max_speed: float = MAX_FLEET_SPEED) -> float:\n    """Calculate fleet speed from ship count using the Orbit Wars curve.\n\n    Args:\n        ships: Number of ships in the fleet.\n        max_speed: Environment maximum fleet speed.\n\n    Returns:\n        Fleet speed in board units per turn.\n    """\n    ships = max(int(ships), 1)\n    return 1.0 + (max_speed - 1.0) * (math.log(ships) / math.log(1000.0)) ** 1.5\n\n\ndef source_reserve(planet: Any) -> int:\n    """Calculate ships to keep on a source planet before launching."""\n    return int(max(5, float(planet.production) * 3))\n\n\ndef point_segment_distance(\n    px: float,\n    py: float,\n    ax: float,\n    ay: float,\n    bx: float,\n    by: float,\n) -> float:\n    """Calculate the shortest distance from a point to a segment."""\n    dx = bx - ax\n    dy = by - ay\n    if dx == 0 and dy == 0:\n        return math.hypot(px - ax, py - ay)\n    t = ((px - ax) * dx + (py - ay) * dy) / (dx * dx + dy * dy)\n    t = max(0.0, min(1.0, t))\n    closest_x = ax + t * dx\n    closest_y = ay + t * dy\n    return math.hypot(px - closest_x, py - closest_y)\n\n\ndef crosses_sun(source: Any, target: Any) -> bool:\n    """Return whether the direct launch segment intersects the sun."""\n    center_x, center_y = BOARD_CENTER\n    clearance = point_segment_distance(\n        center_x,\n        center_y,\n        float(source.x),\n        float(source.y),\n        float(target.x),\n        float(target.y),\n    )\n    return clearance <= SUN_RADIUS + SUN_SAFETY_MARGIN\n\n\ndef ships_to_send(source: Any, target: Any) -> Optional[int]:\n    """Calculate a conservative capture fleet while preserving reserve.\n\n    Args:\n        source: Owned source planet.\n        target: Non-owned target planet.\n\n    Returns:\n        Ship count to send, or `None` if the source cannot afford the move.\n    """\n    needed = int(target.ships) + 1\n    available = int(source.ships) - source_reserve(source)\n    if available < needed:\n        return None\n    return int(needed)\n\n\ndef target_score(source: Any, target: Any, ships: int) -> float:\n    """Score a target by production value adjusted for cost and travel time."""\n    travel_time = distance(source, target) / fleet_speed(ships)\n    capture_cost = max(float(ships), 1.0)\n    production_value = float(target.production) * 30.0\n    enemy_bonus = 20.0 if int(target.owner) >= 0 else 0.0\n    return (production_value + enemy_bonus) / (capture_cost + travel_time)\n\n\ndef best_target(source: Any, targets: list) -> Optional[tuple]:\n    """Choose the best affordable, sun-safe target for one source planet."""\n    best = None\n    best_score = float("-inf")\n    for target in targets:\n        if crosses_sun(source, target):\n            continue\n        ships = ships_to_send(source, target)\n        if ships is None:\n            continue\n        score = target_score(source, target, ships)\n        if score > best_score:\n            best_score = score\n            best = (target, ships)\n    return best\n\n\ndef agent(obs: Any) -> list:\n    """Return ROI-ranked, reserve-safe, sun-safe Orbit Wars launch actions."""\n    try:\n        player = int(get_obs_value(obs, "player", 0))\n        raw_planets = get_obs_value(obs, "planets", [])\n        planets = parse_planets(raw_planets)\n        my_planets = [p for p in planets if int(p.owner) == player]\n        targets = [p for p in planets if int(p.owner) != player]\n\n        moves = []\n        if not targets:\n            return moves\n\n        for source in my_planets:\n            chosen = best_target(source, targets)\n            if chosen is None:\n                continue\n            target, ships = chosen\n            moves.append([int(source.id), launch_angle(source, target), int(ships)])\n        return moves\n    except Exception:\n        return []\n'

CFG = {
    'n_seeds': 30,
    'primary_opponent': 'random',
    'fallback_opponent': 'passive',
    'run_simulations': True,
    'version_name': 'roi_reserve_v1',
}

IS_KAGGLE = Path('/kaggle').exists()
WORKING = Path('/kaggle/working') if IS_KAGGLE else Path('outputs/agent_roi_reserve_v1')
WORKING.mkdir(parents=True, exist_ok=True)

SOURCE_CANDIDATES = [
    Path('main.py'),
    Path('../agents/roi_reserve_v1/main.py'),
    Path('agents/roi_reserve_v1/main.py'),
    Path('/kaggle/working/main.py'),
]


def first_existing(paths: Sequence[Path]) -> Optional[Path]:
    """Return the first existing path from ordered candidates.

    Args:
        paths: Candidate paths in priority order.

    Returns:
        Existing path, or `None` when no path exists.
    """
    for path in paths:
        if path.exists():
            return path
    return None


SOURCE_MAIN = first_existing(SOURCE_CANDIDATES)
SUBMISSION_MAIN = WORKING / 'main.py'
if SOURCE_MAIN is None:
    SUBMISSION_MAIN.write_text(EMBEDDED_AGENT_SOURCE)
elif SOURCE_MAIN.resolve() != SUBMISSION_MAIN.resolve():
    shutil.copy2(SOURCE_MAIN, SUBMISSION_MAIN)

print('is_kaggle:', IS_KAGGLE)
print('working:', WORKING)
print('source_main:', SOURCE_MAIN)
print('submission_main:', SUBMISSION_MAIN)
print('cfg:', json.dumps(CFG, indent=2))

## 2. Static Verification

Compile the generated agent and run a small synthetic observation check before using the Kaggle environment.

In [ ]:
compile_result = None
try:
    compile(SUBMISSION_MAIN.read_text(), str(SUBMISSION_MAIN), 'exec')
    compile_result = 'ok'
except Exception:
    compile_result = traceback.format_exc()
    raise


def load_agent(path: Path) -> Any:
    """Load an agent module from a Python file.

    Args:
        path: Path to a Python file containing `agent(obs)`.

    Returns:
        Loaded Python module.
    """
    spec = importlib.util.spec_from_file_location('submission_agent', path)
    module = importlib.util.module_from_spec(spec)
    if spec.loader is None:
        raise ImportError(f'Cannot load {path}')
    spec.loader.exec_module(module)
    return module


def run_synthetic_action_check(agent_module: Any) -> list:
    """Run a deterministic action-format smoke check.

    Args:
        agent_module: Loaded module with `agent(obs)`.

    Returns:
        Moves returned by the agent.
    """
    obs = {
        'player': 0,
        'planets': [
            [0, 0, 10.0, 10.0, 1.0, 80, 5],
            [1, -1, 13.0, 10.0, 1.0, 12, 1],
            [2, -1, 24.0, 10.0, 2.0, 10, 5],
        ],
        'fleets': [],
        'angular_velocity': 0.03,
        'initial_planets': [],
        'comet_planet_ids': [],
    }
    moves = agent_module.agent(obs)
    assert isinstance(moves, list)
    for move in moves:
        assert isinstance(move, list)
        assert len(move) == 3
        assert isinstance(move[0], int)
        assert isinstance(move[1], float)
        assert isinstance(move[2], int)
        assert move[2] > 0
    return moves


agent_module = load_agent(SUBMISSION_MAIN)
synthetic_moves = run_synthetic_action_check(agent_module)
print('compile_result:', compile_result)
print('synthetic_moves:', synthetic_moves)

## 3. Kaggle Environment Check

Import Orbit Wars from `kaggle_environments`. Local runs may fail this gate; Kaggle runs should pass.

In [ ]:
env_info = {
    'python': sys.version,
    'platform': platform.platform(),
    'is_kaggle': IS_KAGGLE,
    'version_name': CFG['version_name'],
    'compile_result': compile_result,
    'synthetic_moves': synthetic_moves,
}
ENV_AVAILABLE = False
ENV_ERROR = None

try:
    import kaggle_environments
    from kaggle_environments import make

    ENV_AVAILABLE = True
    env_info['kaggle_environments_version'] = getattr(kaggle_environments, '__version__', 'unknown')
except Exception as exc:
    ENV_ERROR = repr(exc)
    env_info['kaggle_environments_error'] = traceback.format_exc()

try:
    if ENV_AVAILABLE:
        make('orbit_wars', configuration={'seed': 0}, debug=True)
        env_info['orbit_wars_available'] = True
except Exception as exc:
    ENV_AVAILABLE = False
    ENV_ERROR = repr(exc)
    env_info['orbit_wars_available'] = False
    env_info['orbit_wars_error'] = traceback.format_exc()

print(json.dumps(env_info, indent=2, default=str))

## 4. Fixed-Seed Benchmark

Run the agent against `random` over the same seed count as the EDA starter baseline. If `random` is unavailable, fall back to a passive no-op opponent so the notebook still exercises the environment.

In [ ]:
PASSIVE_PATH = WORKING / 'passive_agent.py'
PASSIVE_PATH.write_text(
    'def agent(obs):\n'
    '    """Return no actions for fallback benchmark simulations."""\n'
    '    return []\n'
)


def score_from_obs(obs: Any, player: int) -> float:
    """Calculate a final ship-count proxy for one player.

    Args:
        obs: Final observation as dict-like or object-like value.
        player: Player id to score.

    Returns:
        Sum of owned ships on planets and fleets.
    """
    if isinstance(obs, dict):
        planets = obs.get('planets', []) or []
        fleets = obs.get('fleets', []) or []
    else:
        planets = getattr(obs, 'planets', []) or []
        fleets = getattr(obs, 'fleets', []) or []
    planet_score = sum(float(row[5]) for row in planets if int(row[1]) == player)
    fleet_score = sum(float(row[6]) for row in fleets if int(row[1]) == player)
    return planet_score + fleet_score


def summarize_seed(seed: int, env: Any, opponent_label: str) -> dict[str, Any]:
    """Summarize one completed Orbit Wars episode.

    Args:
        seed: Environment seed.
        env: Completed Kaggle environment.
        opponent_label: Opponent used for the run.

    Returns:
        One row of benchmark metrics.
    """
    final_step = env.steps[-1]
    final_obs = final_step[0].observation
    return {
        'seed': seed,
        'opponent': opponent_label,
        'steps': len(env.steps),
        'player0_reward': final_step[0].reward,
        'player0_status': final_step[0].status,
        'player1_reward': final_step[1].reward if len(final_step) > 1 else None,
        'player1_status': final_step[1].status if len(final_step) > 1 else None,
        'player0_score_proxy': score_from_obs(final_obs, 0),
        'player1_score_proxy': score_from_obs(final_obs, 1),
    }


seed_rows = []
run_errors = []
N_SEEDS = int(CFG['n_seeds'])

if ENV_AVAILABLE and CFG['run_simulations']:
    for seed in range(N_SEEDS):
        try:
            env = make('orbit_wars', configuration={'seed': seed}, debug=True)
            opponent_label = str(CFG['primary_opponent'])
            try:
                env.run([str(SUBMISSION_MAIN), opponent_label])
            except Exception:
                opponent_label = str(CFG['fallback_opponent'])
                env = make('orbit_wars', configuration={'seed': seed}, debug=True)
                env.run([str(SUBMISSION_MAIN), str(PASSIVE_PATH)])
            row = summarize_seed(seed, env, opponent_label)
            seed_rows.append(row)
            print('seed', seed, 'reward', row['player0_reward'], 'score_proxy', row['player0_score_proxy'])
        except Exception as exc:
            run_errors.append({'seed': seed, 'error': repr(exc)})
            print('seed failed', seed, repr(exc))
else:
    print('Orbit Wars environment is unavailable in this runtime. Static verification only.')

seed_df = pd.DataFrame(seed_rows)
errors_df = pd.DataFrame(run_errors)
seed_df.to_csv(WORKING / 'agent_v1_seed_summary.csv', index=False)
errors_df.to_csv(WORKING / 'agent_v1_run_errors.csv', index=False)

env_info['seed_count_requested'] = N_SEEDS
env_info['seed_count_completed'] = len(seed_rows)
env_info['run_error_count'] = len(run_errors)
with open(WORKING / 'agent_v1_environment_info.json', 'w') as f:
    json.dump(env_info, f, indent=2, default=str)

wins = int((seed_df['player0_reward'] == 1).sum()) if not seed_df.empty else 0
losses = int((seed_df['player0_reward'] != 1).sum()) if not seed_df.empty else 0
win_rate = wins / len(seed_df) if len(seed_df) else 0.0
findings = f"""# ROI Reserve Agent v1 Findings

- Version: `{CFG['version_name']}`
- Orbit Wars environment available: `{ENV_AVAILABLE}`
- Seeds completed: `{len(seed_rows)}` of `{N_SEEDS}`
- Run errors: `{len(run_errors)}`
- Wins vs random: `{wins}`
- Losses vs random: `{losses}`
- Win rate vs random: `{win_rate:.1%}`

## Strategy Change

- Replaces nearest-target selection with production-aware ROI scoring.
- Keeps source reserves before launch.
- Rejects direct launch paths crossing the sun.
- Preserves single-file `main.py` submission compatibility.
"""
(WORKING / 'agent_v1_findings.md').write_text(findings)
print(findings)

## 5. Submission Package

Create a root-level `submission.tar.gz` containing `main.py`. The same `main.py` can also be submitted directly as a single-file agent.

In [ ]:
submission_tar = WORKING / 'submission.tar.gz'
with tarfile.open(submission_tar, 'w:gz') as tar:
    tar.add(SUBMISSION_MAIN, arcname='main.py')

print('wrote:', SUBMISSION_MAIN)
print('wrote:', submission_tar)
print('wrote:', WORKING / 'agent_v1_environment_info.json')
print('wrote:', WORKING / 'agent_v1_seed_summary.csv')
print('wrote:', WORKING / 'agent_v1_run_errors.csv')
print('wrote:', WORKING / 'agent_v1_findings.md')